# 미분과 역전파 검증
이봉헌 담당. 작은 손계산, NumPy 역전파, 중심차분, TensorFlow 자동미분을 비교한다. HAR 성능 실험과 별개의 수학 검증이다.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise RuntimeError("Open the notebook from the repository or notebooks directory")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))


## 한 뉴런 손계산
입력 x=2, 가중치 w=3, 편향 b=1, 목표 t=10. 예측 wx+b=7, 손실 (예측-t)²/2=4.5. 연쇄법칙으로 dw=-6, db=-3이다. 역전파는 기울기를 구하고 optimizer는 그 기울기로 파라미터를 갱신한다.

In [2]:
from src.math_checks import scalar_example, run_checks
scalar = scalar_example()
print(json.dumps(scalar, indent=2))

{
  "prediction": 7.0,
  "loss": 4.5,
  "gradient_w": -6.0,
  "gradient_b": -3.0,
  "autodiff": [
    -6.0,
    -3.0
  ],
  "updated_loss_by_learning_rate": {
    "0.1": 1.125,
    "0.5": 10.125
  },
  "composition_f_x_y_z": {
    "input": [
      -2,
      5,
      4
    ],
    "f": 12.0,
    "gradients": [
      4.0,
      4.0,
      3.0
    ]
  }
}


학습률 0.1에서는 손실이 1.125로 감소하고 0.5에서는 10.125로 증가한다. 큰 학습률이 항상 빠른 수렴을 뜻하지 않는다.

## 작은 MLP의 전체 파라미터 검사
구조는 2 → 3(tanh) → 2(logits)이며 가중치와 편향을 합쳐 17개다. 평균 교차엔트로피에서 출력 기울기는 (P−Y)/B이고, dW=XᵀD, db=sum(D)로 계산한다. 배치 크기로 두 번 나누지 않는다.

중심차분은 [L(θ+h)−L(θ−h)]/(2h)이다. float64에서 h=10⁻³,10⁻⁴,10⁻⁵를 모두 검사한다. tanh를 사용해 ReLU의 0 지점 미분 문제를 피한다.

In [3]:
summary = run_checks(ROOT / "reports/math")
print(json.dumps(summary, indent=2))
assert summary["passed"]

{
  "passed": true,
  "dtype": "float64",
  "architecture": "2 -> 3 tanh -> 2 logits",
  "parameter_count": 17,
  "loss": 0.6718458139214565,
  "max_relative_error_by_h": {
    "0.001": 8.137703184921284e-07,
    "0.0001": 8.655004280915248e-09,
    "1e-05": 6.2529506532382655e-09
  },
  "max_autodiff_absolute_error": 5.551115123125783e-17,
  "scalar_example": {
    "prediction": 7.0,
    "loss": 4.5,
    "gradient_w": -6.0,
    "gradient_b": -3.0,
    "autodiff": [
      -6.0,
      -3.0
    ],
    "updated_loss_by_learning_rate": {
      "0.1": 1.125,
      "0.5": 10.125
    },
    "composition_f_x_y_z": {
      "input": [
        -2,
        5,
        4
      ],
      "f": 12.0,
      "gradients": [
        4.0,
        4.0,
        3.0
      ]
    }
  }
}


In [4]:
import csv
rows = list(csv.DictReader((ROOT / "reports/math/gradient_details.csv").open()))
print("Checks:", len(rows), "= 17 parameters x 3 step sizes")
for row in rows[:6]:
    print(row)

Checks: 51 = 17 parameters x 3 step sizes
{'h': '0.001', 'parameter': 'W1', 'index': '(0, 0)', 'analytical': '0.004656455935167789', 'finite_difference': '0.004656459305363292', 'autodiff': '0.0046564559351677855', 'relative_error': '3.61884051968256e-07', 'autodiff_absolute_error': '3.469446951953614e-18'}
{'h': '0.001', 'parameter': 'W1', 'index': '(0, 1)', 'analytical': '-0.004983385138876794', 'finite_difference': '-0.00498338657567432', 'autodiff': '-0.004983385138876791', 'relative_error': '1.4415876751180554e-07', 'autodiff_absolute_error': '2.6020852139652106e-18'}
{'h': '0.001', 'parameter': 'W1', 'index': '(0, 2)', 'analytical': '0.002936088332167356', 'finite_difference': '0.0029360884536422382', 'autodiff': '0.0029360883321673556', 'relative_error': '2.0686516545280265e-08', 'autodiff_absolute_error': '4.336808689942018e-19'}
{'h': '0.001', 'parameter': 'W1', 'index': '(1, 0)', 'analytical': '-0.05741357692958312', 'finite_difference': '-0.05741357317018414', 'autodiff': '-

## 설명할 내용
상대 오차의 분모는 max(10⁻⁸, |수치 기울기|+|해석 기울기|)이다. h를 작게 하면 절단 오차는 줄지만 너무 작으면 반올림 오차가 커진다. 일치 확인은 이 작은 예제의 미분 구현 검증이며 전체 학습의 일반화 성능을 입증하지 않는다.